# OpenTorpedo — Teknofest optimization

Runs grid search on [efeerdogmus0/opentorpedo](https://github.com/efeerdogmus0/opentorpedo) in Google Colab.

**Workflow:** Runtime → Run all (four code cells).

| Step | Cell | Action |
|------|------|--------|
| 1 | Code | Clone repository from GitHub |
| 2 | Code | Verify NumPy / SciPy |
| 3 | Code | Run optimization (`PRESET`) |
| 4 | Code | Download `teknofest_optimized.json` |

| `PRESET` | Combinations | ~Runtime |
|----------|--------------|----------|
| `quick` | 96 | 2 min |
| `fast` | 128 | 3 min |
| `medium` | 2,592 | 10 min |
| `full` | 18,432 | 45–90 min |

**Constraints:** mass ≤ 500 g · 4× parallel springs · wire ≤ 2 mm · coil ≤ 16 mm · length ≤ 60 mm · manufacturability filter

In [ ]:
# Step 1/4 — Clone repository
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/opentorpedo")
GIT_URL = "https://github.com/efeerdogmus0/opentorpedo.git"

# Colab: never delete the cwd — move to /content first, then remove old clone
os.chdir("/content")
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

proc = subprocess.run(
    ["git", "clone", "--depth", "1", GIT_URL, str(WORKDIR)],
    cwd="/content",
    capture_output=True,
    text=True,
)
if proc.returncode != 0:
    raise RuntimeError(f"git clone failed:\n{proc.stderr or proc.stdout}")

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

print("Cloned:", WORKDIR)
print("Files:", sorted(p.name for p in WORKDIR.iterdir() if p.is_file())[:10])

In [ ]:
# Step 2/4 — Dependencies
import numpy as np
import scipy
print(f"numpy {np.__version__} | scipy {scipy.__version__}")

In [ ]:
# Step 3/4 — Grid search
from teknofest.grid_search import run_grid_search, count_combos

PRESET = "full"  # quick | fast | medium | full

n = count_combos(PRESET)
print(f"Preset: {PRESET} | {n} combinations\n")

result = run_grid_search(PRESET, root=WORKDIR, progress_every=100)

In [ ]:
# Step 4/4 — Results and download
from IPython.display import display, JSON
from google.colab import files

display(JSON(result))

out = WORKDIR / "configs" / "teknofest_optimized.json"
files.download(str(out))

b, s, h = result["ballast"], result["spring"], result["hull"]
n = s.get("parallel_count", 4)

print("\n" + "=" * 52)
print("DESIGN SUMMARY (simulation)")
print("=" * 52)
print(f"Max speed (sim)      {result['max_speed_m_s']} m/s")
print(f"Total mass           {result['total_mass_g']} g")
print(f"Center of gravity    {result['cog_cm']} cm from nose")
print(f"Hull infill          {h['infill_percent']}%  ({h['mass_g']} g)")
print(f"Ballast              {b['mass_g']} g @ {b['position_cm']} cm")
print(f"Springs              {n}x identical (parallel)")
print(f"Wire diameter        {s['wire_diameter_mm']} mm each")
print(f"Mean coil diameter   {s['coil_diameter_mm']} mm")
print(f"Active coils         {s['active_coils']}")
print(f"Free length          {s['free_length_mm']} mm")
print(f"Compression          {s['compression_mm']} mm")
print(f"Force (total)        {s.get('force_total_N', '—')} N")
print(f"Force (per spring)   {s.get('force_per_spring_N', '—')} N")
print(f"Launch delta-v       {s['delta_v_m_s']} m/s")
print("=" * 52)
print("Save JSON to configs/ on your machine.")